# Session 4: Exercises

**Do these before session 5.** They take about ninety minutes; the ⭐⭐⭐ ones join the session's ideas
together.

Same pattern as before: **predict, then measure.** Write down what you expect before you run a cell.
Predictions and answers go in the cells marked **✍️**; code to write is marked `# TODO`.

Exercises 1, 5 and the first half of 6 need no key. Exercise 2 needs LangSmith; 3 and 4 need Gemini
and Groq.

Worked solutions are released after the next session.

## Setup (just run it)

In [ ]:
import os
import sys
import time
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda

sys.path.insert(0, str(Path.cwd() / "project"))
from config import settings                          # noqa: E402
from schemas import TicketFacts, TriageResult        # noqa: E402
from utils import ask_json, load_inbox               # noqa: E402
import triage as v1                                  # noqa: E402

TICKETS = load_inbox()
by_id = {t.id: t for t in TICKETS}
print(f"{settings.model_id} · fallback {settings.FALLBACK} · {len(TICKETS)} tickets")

## ⭐ Exercise 1: Read the profile

**The idea.** Lesson section 2.6. Before you switch to a model, or trust it as a fallback, check that
it can do what your program needs. `model.profile` is that information as a dictionary, and it can be
`None`.

Write `missing_capabilities(profile, needs)`. `needs` is a set of profile keys that must be `True`,
like `{"image_inputs", "tool_calling"}`. Return a sorted list of the ones the model lacks. If the
profile is `None` or empty, return `["unknown: no profile"]`: missing data is not the same as "fine".

**✍️ Predict first:** ShopWise needs images (session 3's photo ticket) and, from session 5, tool
calling. Which of the five providers from section 3 pass?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
def missing_capabilities(profile, needs):
    # TODO: return a sorted list of the keys in `needs` that aren't True in `profile`.
    #       A profile of None or {} means you can't tell: return ["unknown: no profile"].
    return []

In [ ]:
def check_1():
    full = {"image_inputs": True, "tool_calling": True, "structured_output": True}
    no_images = {"image_inputs": False, "tool_calling": True}
    cases = [
        (full, {"image_inputs", "tool_calling"}, []),
        (no_images, {"image_inputs", "tool_calling"}, ["image_inputs"]),
        ({"tool_calling": True}, {"image_inputs"}, ["image_inputs"]),       # key absent = not supported
        (None, {"tool_calling"}, ["unknown: no profile"]),
        ({}, {"tool_calling"}, ["unknown: no profile"]),
    ]
    for profile, needs, expected in cases:
        got = missing_capabilities(profile, needs)
        if got != expected:
            return f"for {profile} and {needs}: expected {expected}, got {got}"
    return "passed"


print(check_1())

Now use it on real profiles (no API call is made; profiles ship with the packages):

In [ ]:
NEEDS = {"image_inputs", "tool_calling"}
for model_id in ["google_genai:gemini-3.5-flash-lite", "groq:openai/gpt-oss-20b", "openai:gpt-5.4-nano",
                 "anthropic:claude-haiku-4-5-20251001", "ollama:llama3.2:3b", "google_genai:gemini-3.8-flash"]:
    try:
        profile = init_chat_model(model_id).profile
    except Exception as err:               # some integrations want their key even to build the object
        print(f"{model_id:38} couldn't build it: {type(err).__name__}")
        continue
    print(f"{model_id:38} missing: {missing_capabilities(profile, NEEDS) or 'nothing'}")

**✍️ What did you find, and what would you do about the `None` ones?**

## ⭐ Exercise 2: Find it in LangSmith

**The idea.** Lesson section 7. The habit is to open the run before you add a `print`. Run three
tickets through v1 (tracing must be on), then answer from the LangSmith UI only. No printing, no
reading the return values.

In [ ]:
import subprocess

print("tracing:", os.getenv("LANGSMITH_TRACING"), "| project:", os.getenv("LANGSMITH_PROJECT"))
result = subprocess.run([sys.executable, "triage.py", "--limit", "3"], cwd="project",
                        capture_output=True, text=True)
print(result.stdout[:300], "...")

**✍️ From the UI only:**

1. Which of the three tickets took longest, and which of its three steps was the slow one?
2. How many input tokens did `TCK-001`'s `draft_reply` call use, and why is it so much bigger than
   its `TriageResult` call?
3. What did the whole run of three tickets cost at list price?

## ⭐⭐ Exercise 3: The switch, measured field by field

**The idea.** Lesson section 5.1. `score.py` compares one field, the category, with our labels.
Before a switch you also want to know where two models *disagree with each other*, including fields
we never labelled. Triage all twenty tickets with the default model and with Groq, and count the
disagreements per field.

**✍️ Predict first:** which of `category`, `urgency`, `sentiment`, `needs_human` will the two models
disagree on most? Which least, and why?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
FIELDS = ["category", "urgency", "sentiment", "needs_human"]
models = {"gemini": init_chat_model(settings.model_id), "groq": init_chat_model(settings.FALLBACK)}

disagreements = {field: 0 for field in FIELDS}
for ticket in TICKETS:
    messages = [("system", v1.triage_system()), ("human", ticket.as_block())]
    # TODO: triage the ticket with both models (ask_json with TriageResult),
    #       then add 1 to disagreements[field] for every field where they differ.
    pass

print(disagreements)

**✍️ Which field disagreed most? Is that field's disagreement visible to anyone today?**

## ⭐⭐ Exercise 4: Find your rate limit

**The idea.** Lesson section 6.2. The right `REQUESTS_PER_SECOND` belongs to your key, not to the
code. Find it: send twenty tiny requests in a batch at a few different rates, with `max_retries=1` so
a 429 shows up as an error instead of being retried away, and count the failures.

This spends up to sixty requests of your free quota.

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter


def trial(requests_per_second, n=20):
    limiter = InMemoryRateLimiter(requests_per_second=requests_per_second)
    model = init_chat_model(settings.model_id, max_retries=1, rate_limiter=limiter)
    # TODO: send n tiny requests with RunnableLambda(...).batch(..., return_exceptions=True)
    #       and max_concurrency=n. Return (seconds taken, number of results that are exceptions).
    return 0.0, 0


for rate in [4, 1, 0.25]:
    seconds, failures = trial(rate)
    print(f"{rate:5} req/s   {seconds:5.1f}s   {failures} failed")

**✍️ What rate would you put in `.env` for your key, and how long would the whole inbox (60 calls)
take at that rate?**

## ⭐⭐⭐ Exercise 5: A fallback that doesn't hide your bugs

**The idea.** Lesson section 5.2, question 1. `with_fallbacks` falls back on *every* exception,
including a 400 that means your own request is wrong. Write the precise version:

- `should_fall_back(err)`: `True` for a rate limit (429), a server error (5xx) or a timeout; `False`
  for anything that means "your request is wrong" (400, 401, 403, 404).
- `invoke_with_plan_b(primary, backup, messages)`: try the primary; on an error, use the backup only
  if `should_fall_back` says so, otherwise re-raise.

Errors arrive in three shapes (lesson section 3.4): LangChain's standard ones have `is_retryable`;
Gemini's wrapper keeps the HTTP code on `err.__cause__.code`; Google's 5xx `ServerError` and Groq's
errors carry the code directly (`.code` and `.status_code`). The checker raises one of each, no key
needed.

In [ ]:
def should_fall_back(err) -> bool:
    # TODO: 1. if err has is_retryable (LangChain's standard exceptions), use it
    #       2. otherwise find an HTTP code on err, or on err.__cause__ (.code or .status_code)
    #       3. fall back for 429 and 5xx; a TimeoutError of any kind also falls back
    return False


def invoke_with_plan_b(primary, backup, messages):
    # TODO
    return primary.invoke(messages)

In [ ]:
def check_5():
    import groq
    import httpx
    from google.genai import errors as google_errors
    from langchain_core.exceptions import ModelInvalidRequestError, ModelRateLimitError
    from langchain_google_genai.chat_models import ChatGoogleGenerativeAIError

    def gemini_error(code):                       # how langchain-google-genai 4.3.2 wraps a 4xx
        wrapper = ChatGoogleGenerativeAIError(f"Error calling model: {code}")
        wrapper.__cause__ = google_errors.ClientError(code, {"error": {"code": code, "message": "x"}})
        return wrapper

    groq_429 = groq.RateLimitError("rate limited", body=None,
                                   response=httpx.Response(429, request=httpx.Request("POST", "https://x")))
    cases = [
        ("Gemini 429", gemini_error(429), True),
        ("Gemini 400", gemini_error(400), False),
        ("Gemini 503", google_errors.ServerError(503, {"error": {"code": 503, "message": "x"}}), True),
        ("Groq 429", groq_429, True),
        ("standard rate limit", ModelRateLimitError("x"), True),
        ("standard bad request", ModelInvalidRequestError("x"), False),
    ]
    for name, err, expected in cases:
        if should_fall_back(err) != expected:
            return f"{name}: should_fall_back returned {not expected}, expected {expected}"

    def raises(err):
        def run(_):
            raise err
        return RunnableLambda(run)

    backup = RunnableLambda(lambda _: "backup answered")
    if invoke_with_plan_b(raises(gemini_error(429)), backup, []) != "backup answered":
        return "a 429 from the primary didn't reach the backup"
    try:
        invoke_with_plan_b(raises(gemini_error(400)), backup, [])
        return "a 400 was hidden by the backup; it should have been raised"
    except ChatGoogleGenerativeAIError:
        pass
    return "passed"


print(check_5())

**✍️ Why can't `with_fallbacks(..., exceptions_to_handle=...)` express this rule on our pinned Gemini
package, and what changes when you upgrade to langchain-google-genai 4.4.0?**

## ⭐⭐⭐ Exercise 6: The same request, three bills

**The idea.** Lesson section 3.2. Measure what the schema costs on each vendor you can reach, then
shrink the schema and measure again.

Part 1: one ticket, every provider you have, input tokens with and without `TriageResult`. Part 2:
`TriageLean` below has the same fields with no descriptions. Does the price drop, and does the answer
change?

**✍️ Predict first:** on which provider will the descriptions cost the most?

**✍️ Your prediction:**

*(write it here)*

In [ ]:
from typing import Literal
from pydantic import BaseModel


class TriageLean(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other", "product_question",
                      "refund", "return", "shipping", "warranty"]
    urgency: Literal["low", "medium", "high"]
    sentiment: Literal["calm", "frustrated", "angry"]
    needs_human: bool
    summary: str


KEY_NAMES = {"google_genai": "GEMINI_API_KEY", "groq": "GROQ_API_KEY",
             "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
PROVIDERS = [m for m in ["google_genai:gemini-3.5-flash-lite", "groq:openai/gpt-oss-20b",
                         "openai:gpt-5.4-nano", "anthropic:claude-haiku-4-5-20251001"]
             if os.getenv(KEY_NAMES[m.split(":")[0]])]
messages = [("system", v1.triage_system()), ("human", by_id["TCK-005"].as_block())]
print(PROVIDERS)

In [ ]:
for model_id in PROVIDERS:
    model = init_chat_model(model_id)
    # TODO: input tokens for (a) no schema, (b) TriageResult, (c) TriageLean.
    #       Use model.invoke(messages).usage_metadata for (a), and
    #       model.with_structured_output(Schema, include_raw=True).invoke(messages)["raw"].usage_metadata
    #       for (b) and (c). Print all three, and the category each structured call chose.
    #       Groq sometimes rejects its own tool call (lesson 5.1): catch the error and print it.

**✍️ Explain the spread. Which part of the bill is the vendor's choice, and which part is yours?**